In [ ]:
import os
import pickle
import json
import itertools
import time
import glob
import random
import copy
import json
import shutil
import joblib
from joblib import Parallel, delayed

import pywt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pandas as pd

import scipy.io as sio
from scipy.interpolate import interp1d
import scipy.signal as signal

import optuna
from optuna.visualization import plot_optimization_history, plot_intermediate_values, plot_param_importances
from optuna.visualization import plot_contour, plot_slice
optuna.logging.set_verbosity(optuna.logging.WARNING)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [ ]:
dataset = "data_noise_vehicle_temperature"
DOFs = list(range(8))
discretization = 1

In [ ]:
dof_names = [
    "CarBody_Vert",       # 0
    "FrontBogie_Vert",    # 1
    "RearBogie_Vert",     # 2
    "Wheel1_Vert",        # 3
    "Wheel2_Vert",        # 4
    "CarBody_Pitch",      # 5
    "FrontBogie_Pitch",   # 6 (top 1, in the waterfall ablation)
    "RearBogie_Pitch"     # 7
]

dof_name_to_idx, idx_to_dof_name = {}, {}
for dof, dof_name in enumerate(dof_names):
    dof_name_to_idx[dof_name] = dof
    idx_to_dof_name[dof] = dof_name

In [ ]:
def define_save_locations(name='', dataset='', DOFs=list(range(8)), discretization=1):
    add_string = ''
    if 'noise' in dataset: add_string += 'n'
    if 'temperature' in dataset: add_string += 't'
    if 'vehicle' in dataset: add_string += 'v'
    if 'speed' in dataset: add_string += 's'
    if add_string == '': add_string += 'all'

    if DOFs:
        add_string += '_DOF'
        for dof in DOFs: add_string += f'_{dof}'

    # Add the discretization size to the folder and DB name
    add_string += f'_disc{discretization}'

    database_name = f"sqlite:///ttbi_{name}_ablation_{add_string}.db"
    output_dir = f"results_{name}_{add_string}"
    cache_dir = f"data_cache_{add_string}"
    
    return database_name, output_dir, cache_dir

In [ ]:
def set_global_seed(seed=42):
    """
    Locks all random seeds across Python, NumPy, and PyTorch to ensure 
    100% reproducibility of data splits, weight initializations, and training.
    """
    print(f"🔒 Locking global random seed to {seed}...")
    
    # 1. Python Standard Library
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. NumPy
    np.random.seed(seed)
    
    # 3. PyTorch (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # For multi-GPU setups
    
    # 4. PyTorch CuDNN (Forces deterministic GPU operations, disables auto-tuning)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

GLOBAL_SEED = 42
set_global_seed(GLOBAL_SEED)

# Load dataset

In [ ]:
import scipy.io as sio
import numpy as np

def load_ttbi_dataset_v3(filepath, requested_dofs, n_passages=200):
    """
    requested_dofs: A list of integers from 0 to 7 mapping to the specific physical DOFs.
      0: Car Body Vert    (AcelPrimVag row 0)
      1: Front Bogie Vert (AcelPrimVag row 1)
      2: Rear Bogie Vert  (AcelPrimVag row 2)
      3: Wheel 1 Vert     (AcelRodaPrimVag row 0)
      4: Wheel 2 Vert     (AcelRodaPrimVag row 1)
      5: Car Body Pitch   (PitchPrimVag row 0)
      6: Front Bogie Pitch(PitchPrimVag row 1)
      7: Rear Bogie Pitch (PitchPrimVag row 2)
    """    
    dataset_path = os.path.join('data', filepath)
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset folder not found: {dataset_path}")

    X_list = []
    y_list = []
    
    for damage_label in range(0, 61):
        # Format the file name, e.g., 0001.mat
        filename = f"{damage_label+1:04d}.mat"
        filepath = os.path.join(dataset_path, filename)
        
        try:
            mat = sio.loadmat(filepath)
            
            # Navigate the deeply nested MATLAB struct
            # mat['data'] is usually a 1x1 object array, containing the struct fields
            data_struct = mat['data'][0, 0]
            
            # Check how many passages actually exist in this file
            available_passages = data_struct['AcelPrimVag'].shape[1]
            passages_to_load = min(n_passages, available_passages)
            
            for p in range(passages_to_load):
                # Extract the full matrices for this passage
                acel_vag = data_struct['AcelPrimVag'][0, p]      # Shape: (3, seq_len)
                acel_roda = data_struct['AcelRodaPrimVag'][0, p] # Shape: (4, seq_len)
                pitch_vag = data_struct['PitchPrimVag'][0, p]    # Shape: (3, seq_len)
                
                passage_channels = []
                
                for dof in requested_dofs:
                    if dof == 0: passage_channels.append(acel_vag[0, :])
                    elif dof == 1: passage_channels.append(acel_vag[1, :])
                    elif dof == 2: passage_channels.append(acel_vag[2, :])
                    elif dof == 3: passage_channels.append(acel_roda[0, :])
                    elif dof == 4: passage_channels.append(acel_roda[1, :])
                    elif dof == 5: passage_channels.append(pitch_vag[0, :])
                    elif dof == 6: passage_channels.append(pitch_vag[1, :])
                    elif dof == 7: passage_channels.append(pitch_vag[2, :])
                        
                # Stack into (Channels, Sequence_Length)
                X_list.append(np.vstack(passage_channels))
                y_list.append(damage_label)
                
        except FileNotFoundError:
            print(f"  [!] Missing file: {filename}")
        except KeyError:
            print(f"  [!] Field not found in {filename}")
        except Exception as e:
            print(f"  [!] Error processing {filename}: {e}")

    # Convert to PyTorch-friendly NumPy arrays
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64) # int64 is standard for PyTorch classification labels
        
    return X, y

# Preprocessors

In [ ]:
class TTBIPreprocessor:
    def __init__(self, method='raw', n_segments=512, cwt_scales=64):
        """
        Initializes the preprocessor for the TTBI ablation study.
        
        Args:
            method (str): 'raw', 'paa', 'fft', or 'cwt'.
            n_segments (int): The target length for PAA downsampling (or FFT bin count).
            cwt_scales (int): Number of frequency scales for the Wavelet Transform.
        """
        self.method = method.lower()
        self.n_segments = n_segments
        self.cwt_scales = cwt_scales
        # self.scaler = MinMaxScaler(feature_range=(0, 1))
        self.scaler = StandardScaler()
        self.is_fit = False

    def _apply_paa(self, X):
        """Downsamples the sequence length using Piecewise Aggregate Approximation (Interpolation)."""
        samples, channels, length = X.shape
        if length == self.n_segments:
            return X
            
        x_old = np.linspace(0, 1, length)
        x_new = np.linspace(0, 1, self.n_segments)
        
        # Fully vectorized interpolation along the last axis (axis=2)
        f_interp = interp1d(x_old, X, axis=2, kind='linear')
        X_paa = f_interp(x_new)
        return X_paa.astype(np.float32)

    def _apply_fft(self, X):
        """Computes the frequency magnitude spectrum using Fast Fourier Transform."""
        # rfft automatically computes only the positive frequencies for real inputs
        fft_coeffs = np.fft.rfft(X, axis=2)
        fft_mag = np.abs(fft_coeffs)
        
        # If the resulting bins don't match our target, interpolate them
        return self._apply_paa(fft_mag)

    def _apply_cwt(self, X):
        """
        Generates 2D Scalograms using the Continuous Wavelet Transform (Morlet wavelet).
        Note: Applies PAA first to prevent RAM Out-Of-Memory crashes.
        """
        X_downsampled = self._apply_paa(X)
        samples, channels, length = X_downsampled.shape
        scales = np.arange(1, self.cwt_scales + 1)
        
        print(f"  -> Computing CWT in parallel (Shape: {samples}x{channels}x{self.cwt_scales}x{length})...")
        start_time = time.perf_counter()

        # Define a helper function to process one sample (all its channels)
        def process_single_sample(i):
            sample_cwt = np.zeros((channels, self.cwt_scales, length), dtype=np.float32)
            for c in range(channels):
                coeffs, freqs = pywt.cwt(X_downsampled[i, c, :], scales, 'morl')
                sample_cwt[c, :, :] = np.abs(coeffs)
            return sample_cwt
            
        # Run across all CPU cores simultaneously (n_jobs=-1)
        results = Parallel(n_jobs=-1, batch_size='auto')(
            delayed(process_single_sample)(i) for i in range(samples)
        )
        
        end_time = time.perf_counter()
        elapsed_time = end_time - start_time

        print(f"  -> Computed CWT in parallel in {elapsed_time:.4f} seconds")
        
        # Stack the parallel results back into a single tensor
        X_cwt = np.stack(results)
        return X_cwt

    def transform(self, X, fit_scaler=False, fit_indices=None):
        """
        Applies the selected signal processing method and scales the data PER CHANNEL.
        """
        # 1. Signal Processing
        if self.method == 'raw':
            X_processed = X
        elif self.method == 'paa':
            X_processed = self._apply_paa(X)
        elif self.method == 'fft':
            X_processed = self._apply_fft(X)
        elif self.method in ['cwt', 'paa_cwt']:
            X_processed = self._apply_cwt(X)
        else:
            raise ValueError(f"Unknown preprocessing method: {self.method}")

        # 2. Physics-Preserving Scaling (Per-Channel)
        original_shape = X_processed.shape
        samples = original_shape[0]
        channels = original_shape[1]
        
        if len(original_shape) == 3:
            X_transposed = np.transpose(X_processed, (0, 2, 1))
        elif len(original_shape) == 4:
            X_transposed = np.transpose(X_processed, (0, 2, 3, 1))
            
        X_flat_by_channel = X_transposed.reshape(-1, channels)
        
        # Apply the Scaler WITHOUT DATA LEAKAGE
        if fit_scaler:
            if fit_indices is not None:
                # Isolate ONLY the allowed training indices to fit the scaler
                # We reshape back to isolate the sample dimension, slice it, then flatten again
                X_to_fit = X_transposed[fit_indices].reshape(-1, channels)
                self.scaler.fit(X_to_fit)
            else:
                self.scaler.fit(X_flat_by_channel) # Fallback if no indices provided
                
            self.is_fit = True
            
        else:
            if not self.is_fit:
                raise RuntimeError("Scaler has not been fitted yet!")
                
        # Transform the ENTIRE dataset using the safely fitted scaler
        X_scaled_flat = self.scaler.transform(X_flat_by_channel)
            
        # Reshape back to the transposed shape
        X_scaled_transposed = X_scaled_flat.reshape(X_transposed.shape)
        
        # Undo the transpose to get back to PyTorch's expected format
        if len(original_shape) == 3:
            X_scaled = np.transpose(X_scaled_transposed, (0, 2, 1))
        elif len(original_shape) == 4:
            X_scaled = np.transpose(X_scaled_transposed, (0, 3, 1, 2))
            
        return X_scaled.astype(np.float32)

    def save_scaler(self, filepath):
        """Saves the fitted scaler for the Digital Twin online phase."""
        with open(filepath, 'wb') as f:
            pickle.dump(self.scaler, f)
        print(f"Scaler saved to {filepath}")

# Models

In [ ]:
# =====================================================================
# 1. Spatial Embedding Module
# =====================================================================
class Space2Vec(nn.Module):
    """
    Learnable spatial embedding layer (adapted from Time2Vec).
    Grounds the vibration signal to physical coordinates on the bridge.
    """
    def __init__(self, seq_len, out_features=8):
        super(Space2Vec, self).__init__()
        self.seq_len = seq_len
        self.out_features = out_features
        
        self.w_linear = nn.Parameter(torch.randn(1, 1))
        self.p_linear = nn.Parameter(torch.randn(1, 1))
        
        self.w_periodic = nn.Parameter(torch.randn(out_features - 1, 1))
        self.p_periodic = nn.Parameter(torch.randn(out_features - 1, 1))

    def forward(self, x_space):
        # x_space shape: (Batch, 1, Sequence_Length)
        linear = self.w_linear * x_space + self.p_linear
        periodic = torch.sin(self.w_periodic * x_space + self.p_periodic)
        return torch.cat([linear, periodic], dim=1) # Shape: (Batch, out_features, Seq_Len)

# =====================================================================
# 2. Multi-Rate Pooling Module (N-HiTS Style)
# =====================================================================
class MultiRatePooling1D(nn.Module):
    """
    Extracts features at multiple temporal resolutions by sub-sampling 
    the sequence at different pooling rates before the dense layers.
    """
    def __init__(self, pool_rates=[1, 2, 4]):
        super(MultiRatePooling1D, self).__init__()
        self.pool_rates = pool_rates

    def forward(self, x):
        # x shape expects: (Batch, Features, Sequence_Length)
        pooled_outputs = []
        for rate in self.pool_rates:
            if rate > 1:
                pooled = F.max_pool1d(x, kernel_size=rate, stride=rate)
            else:
                pooled = x
            
            # Flatten the pooled sequence 
            pooled_outputs.append(pooled.flatten(start_dim=1))
            
        # Concatenate all temporal resolutions together
        return torch.cat(pooled_outputs, dim=1) # Shape: (Batch, Flattened_Features)

# =====================================================================
# 3. The Ultimate Modular Network
# =====================================================================
class SpaceAwareModularNetwork(nn.Module):
    """
    Fully modular architecture supporting dynamic insertion of Space2Vec, 
    LSTM, and N-HiTS blocks for rigorous physical ablation studies.
    """
    def __init__(self, n_segments, n_classes, in_channels, params, 
                 use_space2vec=True, use_lstm=True, use_nhits=True, s2v_features=8):
        super(SpaceAwareModularNetwork, self).__init__()
        
        self.params = params
        self.use_space2vec = use_space2vec
        self.use_lstm = use_lstm
        self.use_nhits = use_nhits
        self.n_segments = n_segments
        
        # ---------------------------------------------------------
        # 1. Setup Space2Vec
        # ---------------------------------------------------------
        if self.use_space2vec:
            self.space2vec = Space2Vec(seq_len=n_segments, out_features=s2v_features)
            cnn_in_channels = in_channels + s2v_features
        else:
            cnn_in_channels = in_channels
            
        # ---------------------------------------------------------
        # 2. Build Dynamic CNN Layers
        # ---------------------------------------------------------
        self.cnn_layers = nn.ModuleList()
        current_seq_len = n_segments
        
        n_conv_layers = self.params.get('n_conv_layers', 2)
        for i in range(n_conv_layers):
            out_channels = self.params[f'n_filters_l{i}']
            kernel_size = self.params[f'kernel_size_l{i}']
            
            self.cnn_layers.append(nn.Conv1d(cnn_in_channels, out_channels, kernel_size=kernel_size, padding='same'))
            self.cnn_layers.append(nn.ReLU())
            
            if self.params.get(f'pooling_l{i}', False):
                self.cnn_layers.append(nn.MaxPool1d(kernel_size=2, stride=2))
                current_seq_len = current_seq_len // 2
                
            cnn_in_channels = out_channels
            
        # ---------------------------------------------------------
        # 3. Build Optional LSTM Layer
        # ---------------------------------------------------------
        if self.use_lstm:
            lstm_hidden = self.params.get('lstm_hidden_size', 64)
            lstm_layers = self.params.get('lstm_num_layers', 1)
            
            self.lstm = nn.LSTM(
                input_size=cnn_in_channels, 
                hidden_size=lstm_hidden, 
                num_layers=lstm_layers, 
                batch_first=True,
                dropout=self.params.get('lstm_dropout', 0.2) if lstm_layers > 1 else 0.0
            )
            current_features = lstm_hidden
        else:
            current_features = cnn_in_channels

        # ---------------------------------------------------------
        # 4. Build Optional N-HiTS Multi-Rate Pooling
        # ---------------------------------------------------------
        if self.use_nhits:
            # Dynamically grab the pool rates from Optuna (default to [1,2,4])
            pool_rates = self.params.get('nhits_pool_rates', (1, 2, 4))
            self.multi_rate_pool = MultiRatePooling1D(pool_rates=pool_rates)
            
            # Calculate the math for the Flattened dimension dynamically
            flattened_size = 0
            for rate in pool_rates:
                # Add the length of each sub-sampled sequence
                flattened_size += current_features * (current_seq_len // rate)
        else:
            # Fallback to standard Global Average Pooling
            flattened_size = current_features

        # ---------------------------------------------------------
        # 5. Build Dynamic Dense (Classification) Layers
        # ---------------------------------------------------------
        self.dense_layers = nn.ModuleList()
        n_dense_layers = self.params.get('n_dense_layers', 1)
        in_features = flattened_size 
        
        for i in range(n_dense_layers):
            out_features = self.params[f'n_dense_units_l{i}']
            self.dense_layers.append(nn.Linear(in_features, out_features))
            self.dense_layers.append(nn.ReLU())
            self.dense_layers.append(nn.Dropout(self.params.get(f'dropout_l{i}', 0.2)))
            in_features = out_features
            
        self.final_layer = nn.Linear(in_features, n_classes)

    # ---------------------------------------------------------
    # 6. The Forward Pass (Data Routing)
    # ---------------------------------------------------------
    def forward(self, x):
        batch_size = x.size(0)
        
        # 1. Space2Vec Pass
        if self.use_space2vec:
            space_vector = torch.linspace(0, 1, steps=self.n_segments, device=x.device)
            space_vector = space_vector.view(1, 1, -1).expand(batch_size, 1, -1)
            s2v_embeddings = self.space2vec(space_vector)
            x = torch.cat([x, s2v_embeddings], dim=1)
            
        # 2. CNN Pass -> Output shape: (Batch, Channels, SeqLen)
        for layer in self.cnn_layers:
            x = layer(x)
            
        # 3. LSTM Pass
        if self.use_lstm:
            # LSTM expects: (Batch, SeqLen, Features)
            x = x.permute(0, 2, 1) 
            x, _ = self.lstm(x)
            # Pooling expects: (Batch, Features, SeqLen)
            x = x.permute(0, 2, 1) 
            
        # 4. Pooling / Sub-sampling Pass
        if self.use_nhits:
            # N-HiTS outputs a flat 2D tensor: (Batch, Flattened_Features)
            x = self.multi_rate_pool(x)
        else:
            # Global Average Pooling outputs a flat 2D tensor: (Batch, Features)
            x = torch.mean(x, dim=2) 
            
        # 5. Dense Pass
        for layer in self.dense_layers:
            x = layer(x)
            
        return self.final_layer(x)


# =====================================================================
# 4. The 2D CNN for CWT
# =====================================================================
class Simple2DCNN(nn.Module):
    """
    Dynamic 2D CNN designed specifically for Continuous Wavelet Transform (CWT) scalograms.
    Input shape expects: (Batch_Size, Channels, Scales_Height, Sequence_Width)
    """
    def __init__(self, in_channels, n_classes, params, image_height=64, image_width=512):
        super(Simple2DCNN, self).__init__()
        self.params = params
        self.layers = nn.ModuleList()
        
        # Track the spatial dimensions to calculate the flatten size mathematically
        current_h = image_height
        current_w = image_width
        current_channels = in_channels
        
        n_conv_layers = self.params['n_conv_layers']
        
        # 1. Dynamic 2D Convolutional Layers
        for i in range(n_conv_layers):
            out_channels = self.params[f'n_filters_l{i}']
            # Optuna suggests a single integer (e.g., 3), which PyTorch interprets as a (3, 3) square kernel
            k_size = self.params[f'kernel_size_l{i}']
            
            # padding='same' ensures the convolution doesn't shrink the image dimensions
            self.layers.append(nn.Conv2d(current_channels, out_channels, kernel_size=k_size, padding='same'))
            self.layers.append(nn.ReLU())
            
            # 2x2 Max Pooling cuts the height and width exactly in half
            if self.params.get(f'pooling_l{i}', False):
                self.layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
                current_h = current_h // 2
                current_w = current_w // 2
                
            current_channels = out_channels
            
        # 2. Flatten for the Dense Layers
        self.layers.append(nn.Flatten())
        flattened_size = current_channels * current_h * current_w
        
        # 3. Dynamic Dense (Linear) Layers
        n_dense_layers = self.params['n_dense_layers']
        in_features = flattened_size
        
        for i in range(n_dense_layers):
            out_features = self.params[f'n_dense_units_l{i}']
            self.layers.append(nn.Linear(in_features, out_features))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.Dropout(self.params.get(f'dropout_l{i}', 0.2)))
            in_features = out_features
            
        # 4. Final Classification Output
        self.layers.append(nn.Linear(in_features, n_classes))
        
    def forward(self, x):
        """
        Args:
            x: Input scalograms of shape (Batch, Channels, Height, Width)
        """
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
def build_model(config, params, input_shape, device):
    """
    Universal factory function to dynamically build the PyTorch architecture.
    Calculates the number of classes automatically.
    """
    # 1. Calculate the dynamic number of classes
    # If max damage is 60, and discretization is 5, classes = (60 / 5) + 1 = 13.
    # We default to 1 so older configs without this key still default to 61 classes.
    disc = config.get('discretization', 1) 
    n_classes = int(60 / disc) + 1
    
    in_channels = input_shape[1]
    
    # 2. Map N-HiTS strings back to tuples safely if they exist
    if 'nhits_pool_rates_key' in params:
        pool_map = {"1_2_4": (1, 2, 4), "1_4_8": (1, 4, 8), "1_3_6": (1, 3, 6), "1_2_4_8": (1, 2, 4, 8)}
        params['nhits_pool_rates'] = pool_map[params['nhits_pool_rates_key']]

    # 3. Route to the correct architecture
    if config.get('model_type') == '2D_CNN' or config.get('method') == 'PAA_CWT':
        model = Simple2DCNN(
            in_channels=in_channels,
            n_classes=n_classes,
            params=params,
            image_height=input_shape[2],
            image_width=input_shape[3]
        ).to(device)
    else:
        model = SpaceAwareModularNetwork(
            n_segments=input_shape[2],
            n_classes=n_classes,
            in_channels=in_channels,
            params=params,
            use_space2vec=config.get('use_space2vec', False),
            use_lstm=config.get('use_lstm', False),
            use_nhits=config.get('use_nhits', False)
        ).to(device)
        
    return model, n_classes

# Training setup

In [ ]:
class MemmapDataset(torch.utils.data.Dataset):
    def __init__(self, X_memmap, y_memmap, indices):
        self.X = X_memmap
        self.y = y_memmap
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get the true index from the shuffled list
        real_idx = self.indices[idx]
        
        # Load ONLY this specific sample into RAM and convert to PyTorch tensor
        # We use .copy() to ensure it doesn't hold a lock on the memmap array
        x_tensor = torch.tensor(self.X[real_idx].copy()).float()
        y_tensor = torch.tensor(self.y[real_idx].copy()).long()
        
        return x_tensor, y_tensor

In [ ]:
def get_or_create_cache(config, dataset_name, cache_dir):
    """
    Checks if the processed data exists. If yes, loads it via memory map.
    If no, loads raw data, processes it, saves the cache, and returns the memory map.
    """
    # 1. Clean the dataset name (removes folder paths and file extensions)
    # e.g., "data/bridge_signals.csv" -> "bridge_signals"
    clean_dataset_name = os.path.splitext(os.path.basename(dataset_name))[0]
    
    # 2. Build the unique cache name based on DOFs and Preprocessor
    dof_str = "_".join(map(str, config['dofs']))
    cache_filename = os.path.join(cache_dir, f"cache_{clean_dataset_name}_{config['method']}_dofs_{dof_str}_disc{str(config['discretization'])}.npy")
    labels_filename = os.path.join(cache_dir, f"cache_{clean_dataset_name}_{config['method']}_dofs_{dof_str}_labels_disc{str(config['discretization'])}.npy")
    scaler_filename = os.path.join(cache_dir, f"scaler_{clean_dataset_name}_{config['method']}_dofs_{dof_str}_disc{str(config['discretization'])}.pkl")
    
    if os.path.exists(cache_filename) and os.path.exists(labels_filename) and os.path.exists(scaler_filename):
        # FAST PATH (Memory Mapped to save RAM)
        # print(f"  [CACHE HIT] Loading {config['method']} data...")
        X_processed = np.load(cache_filename, mmap_mode='r')
        y = np.load(labels_filename, mmap_mode='r')
        
        # Safely load the scaler
        if os.path.exists(scaler_filename):
            scaler = joblib.load(scaler_filename)
        elif os.path.exists(scaler_filename.replace('.pkl', '.pt')):
            scaler = torch.load(scaler_filename.replace('.pkl', '.pt'))
        else:
            scaler = None # It was the dummy file
    else:
        # SLOW PATH (Cache Miss)
        # SLOW PATH (Cache Miss)
        print(f"  [CACHE MISS] Processing and saving data for {config['method']} (DOFs: {dof_str})...")
        X_raw, y = load_ttbi_dataset_v3(
            filepath=dataset_name, 
            requested_dofs=config['dofs'], 
            n_passages=200
            )
        
        # NEW: Generate the Canonical Training Indices to prevent Data Leakage!
        all_indices = np.arange(len(y))
        canonical_train_idx, _ = train_test_split(all_indices, test_size=0.20, random_state=42)
        
        preprocessor = TTBIPreprocessor(method=config['method'], n_segments=512)
        
        # NEW: Pass the indices to the transform function
        X_processed = preprocessor.transform(X_raw, fit_scaler=True, fit_indices=canonical_train_idx)
        
        # Convert labels from 0-60 into their discretized bins
        # Example: if discretization=5: 
        # Label 5 -> Class 1, Label 10 -> Class 2, ..., Label 60 -> Class 12
        disc = config.get('discretization', 1)
        y_discretized = np.round(y / disc).astype(int)
        
        # Save to disk
        np.save(cache_filename, X_processed)
        np.save(labels_filename, y_discretized)
        
        # ==========================================
        # BULLETPROOF SCALER EXPORT
        # ==========================================
        # 1. Safely try to get the scaler (returns None if it doesn't exist)
        scaler_to_save = getattr(preprocessor, 'scaler', None)
        
        if scaler_to_save is None:
            print(f"  [Warning] No '.scaler' attribute found in TTBIPreprocessor for {config['method']}.")
            # Create an empty dummy file so the FAST PATH 'if' statement doesn't break on the next run
            with open(scaler_filename, 'w') as f: f.write("NO_SCALER_USED")
            scaler = None
            
        else:
            # 2. Check if it's a PyTorch object or a standard Scikit-Learn object
            if isinstance(scaler_to_save, torch.nn.Module) or torch.is_tensor(scaler_to_save):
                print("  [Save] PyTorch scaler detected. Using torch.save()...")
                # Change extension to .pt for PyTorch
                scaler_filename = scaler_filename.replace('.pkl', '.pt') 
                torch.save(scaler_to_save, scaler_filename)
            else:
                print("  [Save] Standard scaler detected. Using joblib.dump()...")
                joblib.dump(scaler_to_save, scaler_filename)
            
            scaler = scaler_to_save
        # ==========================================
        
        print(f"  [CACHE SAVED] Data successfully cached in {cache_dir}.")
        
        # Reload in memory-mapped mode to keep RAM usage identical
        X_processed = np.load(cache_filename, mmap_mode='r')
        y = np.load(labels_filename, mmap_mode='r')
        
        scaler = preprocessor.scaler
        
    return X_processed, y, scaler

In [ ]:
# --- Hardware Setup ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# 1. The Core Training & Evaluation Function
# =====================================================================
def train_and_evaluate(trial, config, dataset_name, n_epochs, cache_dir, output_dir):
    """
    Loads data, builds a dynamically sized model, trains it, and returns validation accuracy.
    Includes Optuna pruning to kill unpromising trials early.
    """
    # 1. Load Data (Will instantly load or create it)
    seed = 42
    set_global_seed(seed)
    X_processed, y, _ = get_or_create_cache(config, dataset_name, cache_dir)
    
    # 2. Train/Test Split
    all_indices = np.arange(len(y))
    train_idx, val_idx = train_test_split(all_indices, test_size=0.20, random_state=seed)
    
    train_loader = DataLoader(
        MemmapDataset(X_processed, y, train_idx), 
        batch_size=32, 
        shuffle=True,
        num_workers=0 # Keep at 0 to avoid multiprocessing lock issues with memory maps
    )
    
    val_loader = DataLoader(
        MemmapDataset(X_processed, y, val_idx), 
        batch_size=32, 
        shuffle=False
    )

    # 3. Suggest Hyperparameters
    params = {
        'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True),
        'n_conv_layers': trial.suggest_int('n_conv_layers', 2, 4),
        'n_dense_layers': trial.suggest_int('n_dense_layers', 1, 3),
    }
    
    # Suggest CNN block parameters
    for i in range(params['n_conv_layers']):
        params[f'n_filters_l{i}'] = trial.suggest_int(f'n_filters_l{i}', 16, 128, step=16)
        params[f'kernel_size_l{i}'] = trial.suggest_categorical(f'kernel_size_l{i}', [2, 3, 5, 7])
        params[f'pooling_l{i}'] = trial.suggest_categorical(f'pooling_l{i}', [True, False])
    
    # Suggest dense block parameters    
    for i in range(params['n_dense_layers']):
        params[f'n_dense_units_l{i}'] = trial.suggest_int(f'n_dense_units_l{i}', 32, 256, step=16)
        params[f'dropout_l{i}'] = trial.suggest_float(f'dropout_l{i}', 0.1, 0.5)
    
    # ---------------------------------------------------------
    # NEW MODULAR CHECK: Only suggest LSTM params if the ablation 
    # configuration explicitly asks for the LSTM block.
    # ---------------------------------------------------------
    if config.get('use_lstm', False):
        params['lstm_hidden_size'] = trial.suggest_int('lstm_hidden_size', 32, 128, step=32)
        params['lstm_num_layers'] = trial.suggest_int('lstm_num_layers', 1, 2)
        if params['lstm_num_layers'] > 1:
            params['lstm_dropout'] = trial.suggest_float('lstm_dropout', 0.1, 0.4)
            
    # ---------------------------------------------------------
    # NEW MODULAR CHECK: N-HiTS Pooling Rates
    # ---------------------------------------------------------
    if config.get('use_nhits', False):
        # Define a dictionary mapping safe strings to the physical tuples
        pool_rate_options = {
            "1_2_4": (1, 2, 4),       # Standard gentle slope
            "1_4_8": (1, 4, 8),       # Aggressive low-frequency isolation
            "1_3_6": (1, 3, 6),       # Odd-numbered frequency intervals
            "1_2_4_8": (1, 2, 4, 8)   # Deep hierarchy (captures 4 distinct frequency bands)
        }
        
        # 1. Optuna safely suggests and stores the String key
        selected_key = trial.suggest_categorical(
            'nhits_pool_rates_key', 
            list(pool_rate_options.keys())
        )
        
        # 2. We extract the actual tuple and save it to params for the PyTorch network
        params['nhits_pool_rates'] = pool_rate_options[selected_key]

    # 4. Model Routing
    model, n_classes = build_model(config, params, X_processed.shape, DEVICE)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    # 5. Training Loop with Pruning
    # Initialize the best Error to infinity (since we want to minimize it)
    best_val_error = float('inf')
    patience = 5  # How many epochs to wait before stopping
    patience_counter = 0
    
    for epoch in range(n_epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        # Step the scheduler at the END of each epoch
        scheduler.step()
            
        # Validation
        model.eval()
        total_absolute_error = 0
        total = 0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                outputs = model(batch_X)
                _, predicted = torch.max(outputs.data, 1)
                
                total += batch_y.size(0)
                # Calculate the class distance instead of strict accuracy
                # total_absolute_error += torch.abs(predicted - batch_y).sum().item()
                total_absolute_error += torch.pow(predicted - batch_y, 2).sum().item()
                
        val_error = total_absolute_error / total
                
        # 1. Optuna Pruning (Kills trials that are worse than the median Error)
        trial.report(val_error, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
        # 2. Standard Early Stopping (Kills trials that have plateaued)
        # We now check if the Error has gone DOWN
        if val_error < best_val_error:
            best_val_error = val_error
            patience_counter = 0  # Reset counter
            
            # Save the model weights for THIS specific trial
            model_save_path = os.path.join(output_dir, f"weights_{config['name']}_trial_{trial.number}.pth")
            torch.save(model.state_dict(), model_save_path)
            
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break # Exit the epoch loop early!

    return best_val_error

# =====================================================================
# 2. Objective Wrapper for Optuna
# =====================================================================
def print_best_callback(study, trial):
    """
    Optuna callback that prints a specialized message ONLY when a new best trial is found.
    """
    # Check if the trial that just finished is the best one in the study
    if study.best_trial.number == trial.number:
        print(f"\n🏆 NEW CHAMPION FOUND! Trial {trial.number}")
        print(f"   --> MSE: {trial.value:.4f}")
        print(f"   --> Parameters: {trial.params}")
        
class Objective:
    def __init__(self, config, dataset_name, n_epochs, cache_dir, output_dir):
        self.config = config
        self.dataset_name = dataset_name
        self.n_epochs = n_epochs
        self.cache_dir = cache_dir
        self.output_dir = output_dir

    def __call__(self, trial):
        return train_and_evaluate(trial, self.config, self.dataset_name, self.n_epochs,
                                  self.cache_dir, self.output_dir)

### Plot confusion matrices functions

In [ ]:
def plot_best_model_confusion_matrix(study_name, config, val_loader, db_path="sqlite:///BO_Additive_Noise_Temp_8DOFs.db"):
    """
    Rebuilds the best model from an Optuna study, evaluates it, and plots the Confusion Matrix.
    """
    # 1. Load the study and get the best hyperparameters
    study = optuna.load_study(study_name=study_name, storage=db_path)
    best_params = study.best_params
    
    print(f"Generating matrix for {study_name} with Accuracy: {study.best_value:.4f}")

    # 2. Rebuild the optimal architecture
    sample_x, _ = next(iter(val_loader))
    model, n_classes = build_model(config, best_params, sample_x.shape, DEVICE)

    # Note: You would normally need to load the best saved model weights here (state_dict) 
    # if you saved them during training. If you didn't save weights, the network is initialized 
    # randomly, and you will need to quickly retrain this single best model for the n_epochs.
    
    # 3. Gather Predictions
    model.eval()
    all_preds = []
    all_trues = []
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_trues.extend(batch_y.cpu().numpy())

    # 4. Plot the Confusion Matrix
    cm = confusion_matrix(all_trues, all_preds, labels=range(61))
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=False, cmap='Blues', cbar=True)
    
    plt.title(f'Confusion Matrix: {study_name.replace("_", " ")}', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Predicted Scour Class', fontsize=12, fontweight='bold')
    plt.ylabel('True Scour Class', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'CM_{study_name}.png', dpi=300)
    plt.show()

In [ ]:
def plot_aggregated_confusion_matrix(all_trues, all_preds, n_classes, study_name, output_dir):
    """
    Dynamically scales the heatmap based on n_classes.
    """
    cm = confusion_matrix(all_trues, all_preds, labels=range(n_classes))
    np.save(os.path.join(output_dir, "DT_cpt_matrix.npy"), cm)

    labels = range(n_classes)
    
    # Scale figure size: 16x14 for 61 classes, smaller for 13 classes
    fig_size = max(8, n_classes * 0.25)
    plt.figure(figsize=(fig_size + 2, fig_size))
    
    annot_labels = np.where(cm > 0, cm.astype(str), "")

    ax = sns.heatmap(cm, annot=annot_labels, fmt='', cmap='Blues', cbar=True,
                     xticklabels=labels, yticklabels=labels, 
                     annot_kws={"size": 8 if n_classes > 30 else 10, "weight": "bold"},
                     linewidths=0.2, linecolor='lightgray')
    
    # Draw the red bounding box
    for i in range(n_classes):
        ax.add_patch(patches.Rectangle((i, i), 1, 1, fill=False, edgecolor='#E74C3C', lw=2))
        
    plt.title(f'Risk Categorization: {study_name.replace("_", " ")}', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('CNN Predicted Severity', fontsize=14, fontweight='bold')
    plt.ylabel('True Physical Severity', fontsize=14, fontweight='bold')
    
    plt.xticks(fontsize=8, rotation=90)
    plt.yticks(fontsize=8, rotation=0)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'Aggregated_CM_{study_name}.png'), dpi=300)
    plt.close() # Always close to save memory!

In [ ]:
def plot_cached_confusion_matrix(study, config, dataset_name, cache_dir, output_dir):
    """
    Evaluates the Champion model from Optuna and plots its true confusion matrix.
    """
    print(f"\n--> Generating Confusion Matrix for Champion: {config['name']}")
    
    # 1. Load Data & Isolate the Canonical Validation Set (Seed 42)
    X_processed, y, _ = get_or_create_cache(config, dataset_name, cache_dir)
    all_indices = np.arange(len(y))
    _, val_idx = train_test_split(all_indices, test_size=0.20, random_state=42)
    
    val_loader = DataLoader(
        MemmapDataset(X_processed, y, val_idx), 
        batch_size=32, 
        shuffle=False
    )
    
    # 2. Build the BLANK optimal architecture
    best_params = study.best_params
    model, n_classes = build_model(config, best_params, X_processed.shape, DEVICE)
    
    # 3. LOAD THE WINNING WEIGHTS (This fixes the Ghost Model!)
    best_trial_num = study.best_trial.number
    weights_path = os.path.join(output_dir, f"weights_{config['name']}_trial_{best_trial_num}.pth")
    
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"CRITICAL: Could not find champion weights at {weights_path}")
        
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.eval() # Lock dropout and batchnorm
    
    # 4. Generate True Predictions
    all_trues = []
    all_preds = []
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(DEVICE)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs.data, 1)
            
            all_trues.extend(batch_y.numpy())
            all_preds.extend(predicted.cpu().numpy())
            
    # 5. Plot using your dynamic function
    plot_aggregated_confusion_matrix(all_trues, all_preds, n_classes, config['name'], output_dir)
    print(f"    ✅ True Confusion Matrix saved successfully.")

# For Robustness Evaluation

In [ ]:
def set_random_seed(seed):
    """Locks down all sources of randomness for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def run_single_training(config, params, X_processed, y, seed, n_epochs):
    """Helper function to build, train, and evaluate a single model. Returns (Accuracy, Error)."""
    set_random_seed(seed)
    set_global_seed(seed)
    
    # 1. SPLIT INDICES, NOT DATA (Memory Safe!)
    all_indices = np.arange(len(y))
    
    train_idx, val_idx = train_test_split(all_indices, test_size=0.20, random_state=seed)
    
    # 2. Create DataLoaders using the custom MemmapDataset
    train_loader = DataLoader(
        MemmapDataset(X_processed, y, train_idx), 
        batch_size=32, 
        shuffle=True,
        num_workers=0
    )
    val_loader = DataLoader(
        MemmapDataset(X_processed, y, val_idx), 
        batch_size=32, 
        shuffle=False
    )
    
    # 2. Build the model
    model, n_classes = build_model(config, params, X_processed.shape, DEVICE)

    # 3. Train the model
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params.get('weight_decay', 1e-4))
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    
    model.train()
    for epoch in range(n_epochs):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X.to(DEVICE))
            loss = criterion(outputs, batch_y.to(DEVICE))
            loss.backward()
            optimizer.step()
        
        scheduler.step()

    # 4. Evaluate Accuracy and Mean Squared Error (Class Distance)
    model.eval()
    correct, total_absolute_error, total_squared_error, total = 0, 0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_y_device = batch_y.to(DEVICE)
            outputs = model(batch_X.to(DEVICE))
            _, predicted = torch.max(outputs.data, 1)
            
            total += batch_y.size(0)
            correct += (predicted == batch_y_device).sum().item()
            # Calculate how many classes away the prediction was
            # The MAE Penalty (Linear)
            total_absolute_error += torch.abs(predicted - batch_y_device).sum().item()
            
            # The MSE Penalty (Exponential / Outlier Alarm)
            total_squared_error += torch.pow(predicted - batch_y_device, 2).sum().item()
            
    accuracy = correct / total
    mae = total_absolute_error / total
    mse = total_squared_error / total
    
    return accuracy, mae, mse

## Robustness tests and plots

In [ ]:
def generate_optuna_robustness_plots(study, config, output_dir='.'):
    print(f"--> Generating Individual Landscape Plots for {config['name']}...")
    
    df = study.trials_dataframe()
    df = df[df['state'] == 'COMPLETE']
    param_cols = [col for col in df.columns if col.startswith('params_')]
    
    if not param_cols:
        print("  [WARNING] No parameters found in DataFrame to plot.")
        return

    # Create a dedicated subfolder just for these slice plots!
    slice_dir = os.path.join(output_dir, "Slice_Plots")
    os.makedirs(slice_dir, exist_ok=True)

    # Plot and save each parameter individually
    for col in param_cols:
        param_name = col.replace('params_', '')
        
        plt.figure(figsize=(8, 6))
        ax = plt.gca()
        
        # Scatter the trials
        sns.scatterplot(data=df, x=col, y='value', ax=ax, color='#3498DB', alpha=0.6, s=80, edgecolor='black')
        
        # Highlight the winning configuration
        best_val = study.best_value
        best_param = study.best_params.get(param_name)
        if best_param is not None:
            ax.scatter([best_param], [best_val], color='#E74C3C', marker='*', s=400, edgecolor='black', zorder=5, label='Best Config')

        # Formatting
        plt.title(f"Sensitivity: {param_name}\n({config['name']})", fontweight='bold', fontsize=16, pad=15)
        plt.xlabel(f"{param_name} Value", fontweight='bold', fontsize=14)
        plt.ylabel("Validation Error (Lower is Better)", fontweight='bold', fontsize=14)
        
        if param_name in ['lr', 'weight_decay']:
            plt.xscale('log')
            
        plt.grid(True, alpha=0.3)
        if best_param is not None:
            plt.legend()

        plt.tight_layout()
        
        # Save to the new subfolder!
        save_path = os.path.join(slice_dir, f"Slice_{param_name}.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
    print(f"--> Saved {len(param_cols)} individual Slice Plots to {slice_dir}\n")

In [ ]:
def evaluate_stochastic_robustness(study, config, dataset_name, n_epochs=50, physical_error_tolerance=10.0, cache_dir='', output_dir=''):
    """Runs ONLY the n-seed test and generates a distribution boxplot."""
    print(f"\n--> Evaluating Stochastic Robustness for: {config['name']}...")
    
    # 1. Dynamically calculate the MSE threshold!
    disc = config.get('discretization', 1.0)
    max_error_threshold = (physical_error_tolerance / disc) ** 2
    
    # 2. The Gatekeeper
    if study.best_value > max_error_threshold:
        print(f"  [SKIP] Best MSE ({study.best_value:.2f}) > Threshold ({max_error_threshold:.2f}).")
        print(f"         (Model is off by > {physical_error_tolerance}% physical damage). Skipping 30-seed test.")
        return None

    best_params = study.best_params
    X_processed, y, _ = get_or_create_cache(config, dataset_name, cache_dir)
    json_stoch_file = os.path.join(output_dir, f"robustness_stochastic.json")

    if os.path.exists(json_stoch_file):
        # print("  [CACHE HIT] Loading n-Seed results from disk...")
        with open(json_stoch_file, 'r') as f:
            stoch_data = json.load(f)
            seed_accuracies, seed_maes, seed_mses = stoch_data['accuracies'], stoch_data['maes'], stoch_data['mses']
    else:
        print("  Running n-Seed Stochastic Validation...")
        seed_accuracies, seed_maes, seed_mses = [], [], []
        n = 30
        for run in range(n):
            current_seed = 42 + run 
            val_acc, val_mae, val_mse = run_single_training(config, best_params, X_processed, y, seed=current_seed, n_epochs=n_epochs)
            seed_accuracies.append(val_acc); seed_maes.append(val_mae); seed_mses.append(val_mse)
            
        with open(json_stoch_file, 'w') as f:
            json.dump({'accuracies': seed_accuracies, 'maes': seed_maes, 'mses': seed_mses}, f)

    # =========================================================================
    # 3. Generate and Save the Boxplots
    # =========================================================================
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Stochastic Robustness (30 Monte Carlo Seeds) - {config['name']}", fontsize=14, fontweight='bold', y=1.05)

    # Plot 1: MSE
    sns.boxplot(y=seed_mses, ax=axes[0], color='lightcoral', width=0.4, fliersize=0)
    sns.stripplot(y=seed_mses, ax=axes[0], color='darkred', alpha=0.6, jitter=True, size=5)
    axes[0].set_title('Mean Squared Error (MSE)', fontweight='bold')
    axes[0].set_ylabel('MSE')
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)

    # Plot 2: MAE
    sns.boxplot(y=seed_maes, ax=axes[1], color='lightskyblue', width=0.4, fliersize=0)
    sns.stripplot(y=seed_maes, ax=axes[1], color='darkblue', alpha=0.6, jitter=True, size=5)
    axes[1].set_title('Mean Absolute Error (MAE)', fontweight='bold')
    axes[1].set_ylabel('MAE')
    axes[1].grid(axis='y', linestyle='--', alpha=0.7)

    # Plot 3: Accuracy
    sns.boxplot(y=seed_accuracies, ax=axes[2], color='lightgreen', width=0.4, fliersize=0)
    sns.stripplot(y=seed_accuracies, ax=axes[2], color='darkgreen', alpha=0.6, jitter=True, size=5)
    axes[2].set_title('Strict Accuracy', fontweight='bold')
    axes[2].set_ylabel('Accuracy')
    axes[2].grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plot_path = os.path.join(output_dir, f"Stochastic_Boxplot_{config['name']}.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close() # CRITICAL: Closes the figure so your RAM doesn't fill up over 17 loops
    # =========================================================================

    # 4. Calculate metrics
    mu_mse = np.mean(seed_mses)
    std_mse = np.std(seed_mses)
    ucb_mse = mu_mse + (2 * std_mse)

    # 5. Build scorecard
    scorecard = {
        'Optuna_Lucky_Score': study.best_value,
        'Stochastic_Mean_MSE': mu_mse,
        'Stochastic_Std_MSE': std_mse,
        'UCB_95_MSE': ucb_mse
    }
    
    print(f"  --> UCB 95% MSE: {ucb_mse:.2f}")
    return scorecard

In [ ]:
def plot_stochastic_summary(path_list, db_storage, experiment_root_dir, summary_output_dir, file_prefix='stochastic_summary'):
    print(f"\n--> Generating Stochastic Summary Plots (MSE, MAE, Accuracy)...")
    
    names, mses_list, maes_list, accs_list = [], [], [], []
    optuna_mses, optuna_maes, optuna_accs = [], [], []
    ucb_mses = []

    for step in path_list: 
        study_name = step['name']
        try:
            study = optuna.load_study(study_name=study_name, storage=db_storage)
            # Fetch the primary metric (MSE) and secondary metrics (if available)
            o_mse = study.best_value
            o_mae = study.best_trial.user_attrs.get("MAE", None)
            o_acc = study.best_trial.user_attrs.get("Accuracy", None)
        except KeyError:
            continue
        
        output_dir = os.path.join(experiment_root_dir, study_name) 
        stoch_file = os.path.join(output_dir, 'robustness_stochastic.json')
        
        if os.path.exists(stoch_file):
            with open(stoch_file, 'r') as f:
                data = json.load(f)
                mses, maes, accs = data.get('mses', []), data.get('maes', []), data.get('accuracies', [])
                
            if mses and maes and accs:
                names.append(study_name.replace("_", " "))
                mses_list.append(mses)
                maes_list.append(maes)
                accs_list.append(accs)
                
                optuna_mses.append(o_mse)
                optuna_maes.append(o_mae)
                optuna_accs.append(o_acc)
                
                # Calculate UCB for MSE
                ucb_mses.append(np.mean(mses) + (2 * np.std(mses)))

    if not names:
        print("  [Abort] No valid robustness data found to plot.")
        return

    os.makedirs(summary_output_dir, exist_ok=True)
    x_coords = np.arange(len(names))

    # Helper function to keep the plotting code DRY
    def create_boxplot(data_list, optuna_scores, ylabel, title, filename, is_log=False, ucb_scores=None):
        plt.figure(figsize=(16, 8))
        sns.boxplot(data=data_list, color="#3498DB", width=0.5, fliersize=4, 
                    boxprops=dict(edgecolor='black', alpha=0.8), medianprops=dict(color='black', linewidth=2))

        # Plot Optuna Gold Star if we have the data
        if None not in optuna_scores:
            plt.scatter(x_coords, optuna_scores, color='gold', marker='*', s=400, edgecolor='black', zorder=10, label='Optuna Baseline')

        # Plot UCB Diamond (Only for MSE)
        if ucb_scores:
            plt.scatter(x_coords, ucb_scores, color='#E74C3C', marker='D', s=100, edgecolor='black', zorder=10, label='95% UCB (Risk Ceiling)')
            for i, ucb_val in enumerate(ucb_scores):
                plt.text(i, ucb_val * 1.15 if is_log else ucb_val + (max(ucb_scores)*0.05), f'{ucb_val:.2f}', 
                         ha='center', va='bottom', fontsize=10, fontweight='bold', color='#E74C3C')

        plt.title(title, fontsize=16, fontweight='bold', pad=20)
        plt.ylabel(ylabel, fontsize=14, fontweight='bold')
        plt.xticks(ticks=x_coords, labels=names, rotation=30, ha='right', fontsize=11)
        
        if is_log: plt.yscale('log')
        if "Accuracy" in title: plt.ylim(0, 1.05)
            
        plt.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
        plt.legend(loc='upper right', fontsize=12, framealpha=0.9, edgecolor='black')
        plt.tight_layout()
        plt.savefig(os.path.join(summary_output_dir, filename), dpi=300, bbox_inches='tight')
        plt.close()

    # Generate the 3 plots!
    create_boxplot(mses_list, optuna_mses, 'Mean Squared Error (Log Scale)', 'Architecture Robustness: MSE', f"{file_prefix}_MSE.png", is_log=True, ucb_scores=ucb_mses)
    create_boxplot(maes_list, optuna_maes, 'Mean Absolute Error (Classes Off)', 'Architecture Robustness: MAE', f"{file_prefix}_MAE.png")
    create_boxplot(accs_list, optuna_accs, 'Validation Accuracy', 'Architecture Robustness: Accuracy', f"{file_prefix}_Accuracy.png")

    print(f"--> Successfully saved MSE, MAE, and Accuracy summary plots to {summary_output_dir}")

In [ ]:
def evaluate_parametric_robustness(study, config, dataset_name, baseline_mse, n_epochs=50, cache_dir='', output_dir=''):
    """Runs ONLY the Todd Hyperparameter Perturbation test."""
    print(f"\n--> Evaluating Parametric Robustness (Todd Test) for: {config['name']}...")
    
    best_params = study.best_params
    X_processed, y, _ = get_or_create_cache(config, dataset_name, cache_dir)
    json_sens_file = os.path.join(output_dir, f"robustness_sensitivity.json")

    if os.path.exists(json_sens_file):
        # print("  [CACHE HIT] Loading Parametric Sensitivity results from disk...")
        with open(json_sens_file, 'r') as f:
            perturbation_results = json.load(f)
    else:
        print("  Running Dynamic Hyperparameter Perturbations...")
        params_to_test = {'lr': best_params['lr'], 'weight_decay': best_params.get('weight_decay', 1e-4)}
        for p in ['dropout_l0', 'n_filters_l0', 'lstm_hidden_size']:
            if p in best_params: params_to_test[p] = best_params[p]

        perturbation_results = {}
        for param_name, base_val in params_to_test.items():
            perturbation_results[param_name] = {}
            for mult in [0.9, 0.95, 1.0, 1.05, 1.1]: # Tighter bounds for Todd test
                perturbed_params = copy.deepcopy(best_params)
                new_val = max(1, int(base_val * mult)) if isinstance(base_val, int) else base_val * mult
                perturbed_params[param_name] = new_val
                
                val_acc, val_mae, val_mse = run_single_training(config, perturbed_params, X_processed, y, seed=42, n_epochs=n_epochs)
                perturbation_results[param_name][f"{mult*100:.0f}%"] = {'acc': val_acc, 'mae': val_mae, 'mse': val_mse}

        with open(json_sens_file, 'w') as f:
            json.dump(perturbation_results, f)

    # Calculate max degradation against the stochastic baseline
    worst_perturbed_mse = 0
    for param_data in perturbation_results.values():
        for metrics in param_data.values():
            if metrics['mse'] > worst_perturbed_mse:
                worst_perturbed_mse = metrics['mse']
                
    max_degradation = worst_perturbed_mse - baseline_mse
    
    print(f"  --> Worst Parametric Degradation: +{max_degradation:.2f} MSE")
    return worst_perturbed_mse, max_degradation

In [ ]:
def plot_parametric_summary(champion_name, experiment_root_dir, summary_output_dir):
    print(f"\n--> Generating Parametric Sensitivity Plots for Champion: {champion_name}")
    
    champion_dir = os.path.join(experiment_root_dir, champion_name)
    sens_file = os.path.join(champion_dir, 'robustness_sensitivity.json')
    
    if not os.path.exists(sens_file):
        print(f"  [Abort] Sensitivity JSON not found at {sens_file}")
        return
        
    with open(sens_file, 'r') as f:
        perturbation_results = json.load(f)
        
    os.makedirs(summary_output_dir, exist_ok=True)
        
    # Create a unified plot for each hyperparameter
    for param_name, mult_data in perturbation_results.items():
        x_labels = list(mult_data.keys())
        acc_vals = [d['acc'] for d in mult_data.values()]
        mae_vals = [d['mae'] for d in mult_data.values()]
        mse_vals = [d['mse'] for d in mult_data.values()]
        
        # Setup a 1x3 horizontal layout
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Shared formatting function
        def format_axis(ax, y_vals, title, ylabel, optimal_idx=2):
            ax.plot(x_labels, y_vals, marker='o', linestyle='-', color='#E74C3C', linewidth=2)
            ax.axvline(x=x_labels[optimal_idx], color='black', linestyle='--', label="Optuna Baseline (100%)")
            ax.set_title(title, fontweight='bold', fontsize=12)
            ax.set_xlabel("Parameter Multiplier", fontweight='bold')
            ax.set_ylabel(ylabel, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.legend()
        
        # 1. Accuracy Plot
        format_axis(axes[0], acc_vals, f"Accuracy vs {param_name}", "Accuracy")
        axes[0].set_ylim(0, 1.0)
        
        # 2. MAE Plot
        format_axis(axes[1], mae_vals, f"MAE vs {param_name}", "Classes Off (MAE)")
        
        # 3. MSE Plot
        format_axis(axes[2], mse_vals, f"MSE vs {param_name}", "Squared Error (MSE)")
        
        plt.suptitle(f"Parametric Sensitivity (Todd Test): {champion_name} - {param_name}", fontsize=16, fontweight='bold', y=1.05)
        plt.tight_layout()
        
        save_path = os.path.join(summary_output_dir, f"Parametric_Sensitivity_{param_name}.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
    print(f"--> Saved all parametric summary plots to {summary_output_dir}")

In [ ]:
def export_digital_twin_package(study, config, dataset_name, cache_dir, output_dir):
    """
    Creates the final deployment package for the Digital Twin online framework.
    Assumes the ONNX export and CPT matrix are already saved by other functions.
    """
    print(f"  --> Bundling Digital Twin Package for {config['name']}...")
    
    # 1. Save the Model DNA (Architecture + Best Params + DOFs)
    dt_metadata = {
        "model_name": config['name'],
        "preprocessing_method": config['method'],
        "active_dofs": config['dofs'],
        "architecture_flags": {
            "use_space2vec": config['use_space2vec'],
            "use_lstm": config['use_lstm'],
            "use_nhits": config['use_nhits'],
            "model_type": config['model_type']
        },
        "optimal_hyperparameters": study.best_params
    }
    
    with open(os.path.join(output_dir, "DT_metadata.json"), 'w') as f:
        json.dump(dt_metadata, f, indent=4)
        
    # 2. Copy the correct scaler from the global cache into this model's specific folder
    clean_dataset_name = os.path.splitext(os.path.basename(dataset_name))[0]
    
    dof_str = "_".join(map(str, config['dofs']))
    scaler_source = os.path.join(cache_dir, f"scaler_{clean_dataset_name}_{config['method']}_dofs_{dof_str}.pkl")
    scaler_destination = os.path.join(output_dir, "DT_scaler.pkl")
    
    if os.path.exists(scaler_source):
        shutil.copy(scaler_source, scaler_destination)

# ABLATION

In [ ]:
def execute_ablation_pipeline(experiment_path, database_name, output_dir_name, cache_dir_name, dataset, n_trials=50, epochs=50):
    """
    Master execution loop for running Optuna grids, extracting metrics, 
    and performing robustness stress-tests.
    
    Returns:
        A list of dictionaries containing the robustness scorecard for all models that passed the gatekeeper.
    """
    # Ensure cache directory exists globally
    os.makedirs(cache_dir_name, exist_ok=True)
    
    set_global_seed(42)
    
    all_model_results = []
    
    for step in experiment_path:
        print(f"\n{'='*50}")
        print(f"Executing: {step['name']}")
        print(f"{'='*50}")
        
        # Create the specific folder for this model
        output_dir = os.path.join(output_dir_name, step['name'])
        os.makedirs(output_dir, exist_ok=True)
        
        sampler = optuna.samplers.TPESampler(seed=GLOBAL_SEED)

        study = optuna.create_study(
            study_name=step['name'],
            storage=database_name,
            direction='minimize',
            load_if_exists=True,
            sampler=sampler
        )
        
        # 1. OPTIMIZATION
        if len(study.trials) < n_trials:
            objective = Objective(config=step, dataset_name=dataset, n_epochs=epochs, 
                                  cache_dir=cache_dir_name, output_dir=output_dir)
            study.optimize(
                objective, 
                n_trials=(n_trials - len(study.trials)), 
                callbacks=[print_best_callback]
            )
        
        # 2. EVALUATION & REPORTING (Includes ONNX Export)
        plot_cached_confusion_matrix(study=study, config=step, dataset_name=dataset, 
                                     cache_dir=cache_dir_name, output_dir=output_dir)
        
        export_digital_twin_package(study=study, config=step, dataset_name=dataset,
                                    cache_dir=cache_dir_name, output_dir=output_dir)
        
        # 3. STOCHASTIC STRESS-TEST
        stochastic_scorecard = evaluate_stochastic_robustness(
            study=study, config=step, dataset_name=dataset, 
            n_epochs=epochs, cache_dir=cache_dir_name, output_dir=output_dir
        )
        
        if stochastic_scorecard:
            all_model_results.append({
                'Model': step['name'],
                **stochastic_scorecard
            })
        
        generate_optuna_robustness_plots(study=study, config=step, output_dir=output_dir)
        
        print(f"Best Val Error for {step['name']}: {study.best_value:.4f} MSE")
        
    return all_model_results

## Build architecture ablation grid

In [ ]:
# Initialize the empty grid and a counter for clean naming
ablation_grid_path = []
counter = 1

# =========================================================
# 1. GENERATE THE 16 1D MODELS (RAW & PAA)
# =========================================================
preprocessors_1d = ['RAW', 'PAA']
booleans = [True, False]

# itertools.product creates every possible combination of True/False automatically
for prep in preprocessors_1d:
    for space, lstm, nhits in itertools.product(booleans, repeat=3):
        
        # Build a clean, readable name for the model
        features = []
        if space: features.append("S2V")
        if lstm: features.append("LSTM")
        if nhits: features.append("NHiTS")
        
        feature_str = "_".join(features) if features else "Base"
        name = f"{counter}_{prep}_{feature_str}"
        
        ablation_grid_path.append({
            "name": name,
            "method": prep,
            "dofs": DOFs,
            "discretization": discretization,
            "use_space2vec": space,
            "use_lstm": lstm,
            "use_nhits": nhits,
            "model_type": "1D_MODULAR"
        })
        counter += 1

# =========================================================
# 2. APPEND THE SINGLE 2D MODEL (CWT)
# =========================================================
ablation_grid_path.append({
    "name": f"{counter}_CWT_2D_CNN",
    "method": "PAA_CWT",
    "dofs": DOFs,
    "discretization": discretization,
    "use_space2vec": False,
    "use_lstm": False,
    "use_nhits": False,
    "model_type": "2D_CNN"
})

print(f"Successfully generated {len(ablation_grid_path)} unique architectures for ablation.")

## Architectures ablation pipeline execution

In [ ]:
database_arch, output_arch, cache_arch = define_save_locations(
    'architectures', dataset=dataset, DOFs=DOFs, discretization=discretization
    )

architecture_results = execute_ablation_pipeline(
    experiment_path=ablation_grid_path, 
    database_name=database_arch, 
    output_dir_name=output_arch, 
    cache_dir_name=cache_arch, 
    dataset=dataset,
    n_trials=50,
    epochs=50
)

In [ ]:
plot_stochastic_summary(
    path_list=ablation_grid_path, 
    db_storage=database_arch, 
    experiment_root_dir=output_arch, 
    summary_output_dir=os.path.join(output_arch, "Summary_Plots"),
    file_prefix='architecture_ablation'
)

### Champion selection & Parametric stress test

In [ ]:
# 1. Autopilot: Select the safest architecture
champion_result = min(architecture_results, key=lambda x: x['UCB_95_MSE'])
champion_name = champion_result['Model']
print(f"\n🏆 PROCEEDING WITH CHAMPION ARCHITECTURE: {champion_name}")

# 2. Run the Todd Parametric test on the Champion
champion_config = next(c for c in ablation_grid_path if c['name'] == champion_name)
champion_study = optuna.load_study(study_name=champion_name, storage=database_arch)
champion_dir = os.path.join(output_arch, champion_name)

worst_mse, max_degradation = evaluate_parametric_robustness(
    study=champion_study, 
    config=champion_config, 
    dataset_name=dataset, 
    baseline_mse=champion_result['Stochastic_Mean_MSE'],
    n_epochs=50, 
    cache_dir=cache_arch, 
    output_dir=champion_dir
)

In [ ]:
plot_parametric_summary(
    champion_name=champion_name,
    experiment_root_dir=output_arch,
    summary_output_dir=os.path.join(output_arch, "Summary_Plots")
)

## Leave-One-Out Ablation by removing the DOFs from the best model (check sensor importance)

In [ ]:
DOFs_waterfall_path = []

# 2. Build the new Baseline using the Champion's DNA
DOFs_waterfall_path.append({
    "name": f"Baseline_All_8_{champion_name}",
    "method": champion_config['method'],
    "use_lstm": champion_config['use_lstm'],
    "use_nhits": champion_config['use_nhits'],
    "use_space2vec": champion_config['use_space2vec'],
    "model_type": champion_config['model_type'],
    "dofs": DOFs.copy(),
    "discretization": discretization,
})

# 3. Backward Elimination (Drop exactly one sensor)
for i in range(len(DOFs)):
    current_dofs = DOFs.copy()
    dropped_dof = current_dofs.pop(i)
    
    DOFs_waterfall_path.append({
        "name": f"Drop_{dof_names[dropped_dof]}",
        "method": champion_config['method'],
        "use_lstm": champion_config['use_lstm'],
        "use_nhits": champion_config['use_nhits'],
        "use_space2vec": champion_config['use_space2vec'],
        "model_type": champion_config['model_type'],
        "dofs": current_dofs,
        "discretization": discretization,
    })

print(f"\nGenerated {len(DOFs_waterfall_path)} configurations for Backward Elimination.")
for config in DOFs_waterfall_path:
    print(f" - {config['name']} (Using {len(config['dofs'])} DOFs)")

In [ ]:
database_dofs, output_dofs, cache_dofs = define_save_locations(
    'DOFs_sensitivity', dataset=dataset, DOFs=DOFs, discretization=discretization
    )

dofs_results = execute_ablation_pipeline(
    experiment_path=DOFs_waterfall_path, 
    database_name=database_dofs, 
    output_dir_name=output_dofs, 
    cache_dir_name=cache_dofs, 
    dataset=dataset,
    n_trials=50,
    epochs=50
)

In [ ]:
plot_stochastic_summary(
    path_list=DOFs_waterfall_path, 
    db_storage=database_dofs, 
    experiment_root_dir=output_dofs, 
    summary_output_dir=os.path.join(output_dofs, "Summary_Plots"),
    file_prefix='sensor_elimination'
)

### Ranking the Sensors

In [ ]:
### Determine the winner
# 1. Filter out the Baseline, keep only the "Drop_X" models
drop_results = [res for res in dofs_results if "Drop_" in res['Model']]

# 2. Sort descending by Global Risk Score (Highest Error = Most Important Sensor)
drop_results.sort(key=lambda x: x['UCB_95_MSE'], reverse=True)

ranked_dofs = []
print("\n🏆 SENSOR IMPORTANCE RANKING (Based on Phase 3 Degradation):")
for i, res in enumerate(drop_results):
    sensor_name = res['Model'].replace("Drop_", "")
    ranked_dofs.append(dof_name_to_idx[sensor_name])
    print(f"  {i+1}. {sensor_name} (Caused 95% CI MSE to hit {res['UCB_95_MSE']:.2f} when removed)")

## Cross-Architecture Forward Selection Sweep

In [ ]:
# =====================================================================
# PHASE 4: CROSS-ARCHITECTURE FORWARD SELECTION SWEEP
# =====================================================================
cumulative_dofs = []
forward_sweep_master_results = {}

print("\n" + "="*60)
print("🚀 INITIATING GRAND FINALE: FORWARD SELECTION SWEEP")
print("="*60)

try:
    for i, next_dof in enumerate(ranked_dofs):
        cumulative_dofs.append(next_dof)
        num_sensors = len(cumulative_dofs)
        
        phase_name = f"ForwardSweep_{num_sensors}Sensors"
        
        print(f"\n\n{'*'*50}")
        print(f"🌟 RUNNING SWEEP WITH {num_sensors} SENSORS")
        print(f"   Active DOFs: {cumulative_dofs}")
        print(f"{'*'*50}")
        
        # 1. Generate the grid safely using DEEPCOPY
        current_grid = copy.deepcopy(ablation_grid_path)
        for idx in range(len(current_grid)):
            # Pass a .copy() of the list so it locks in the current state!
            current_grid[idx]["dofs"] = cumulative_dofs.copy()
        
        # 2. Define isolated save locations so databases don't mix
        # (Assuming your define_save_locations accepts a string prefix)
        db_sweep, out_sweep, cache_sweep = define_save_locations(
            phase_name, dataset=dataset, DOFs=cumulative_dofs, discretization=discretization
            )
        
        # 3. Execute the standard model pipeline
        sweep_results = execute_ablation_pipeline(
            experiment_path=current_grid, 
            database_name=db_sweep, 
            output_dir_name=out_sweep, 
            cache_dir_name=cache_sweep, 
            dataset=dataset,
            n_trials=50,
            epochs=50
        )
        
        # Save results to our master dictionary
        forward_sweep_master_results[num_sensors] = sweep_results
        
        # 4. Generate the plots for this specific sensor count
        plot_stochastic_summary(
            path_list=current_grid, 
            db_storage=db_sweep, 
            experiment_root_dir=out_sweep, 
            summary_output_dir=os.path.join(out_sweep, "Summary_Plots"),
            file_prefix=f'forward_sweep_{num_sensors}_sensors'
        )
        
        print(f"✅ Finished Sweep for {num_sensors} Sensors.")

except KeyboardInterrupt:
    print("\n\n⚠️ MANUAL INTERRUPT DETECTED.")
    print(f"Gracefully stopping the Forward Sweep. Completed up to {len(cumulative_dofs)-1} sensors.")
    print("All generated data, databases, and plots up to this point are safely saved!")
    
print("\n🎉 ENTIRE PIPELINE COMPLETE.")

## Isolated single-sensor sweep

In [ ]:
# =====================================================================
# PHASE 5: ISOLATED SINGLE-SENSOR SWEEP
# =====================================================================

isolated_sweep_master_results = {}

print("\n" + "="*60)
print("🔬 INITIATING PHASE 5: ISOLATED MARGINAL UTILITY SWEEP")
print("="*60)

try:
    for rank_idx, current_dof in enumerate(ranked_dofs):
        sensor_name = idx_to_dof_name[current_dof]
        isolated_dof_list = [current_dof] # The network ONLY sees this one sensor!
        
        phase_name = f"IsolatedSweep_{sensor_name}"
        
        print(f"\n\n{'*'*50}")
        print(f"🌟 RUNNING ISOLATED SWEEP FOR: {sensor_name} (Rank #{rank_idx + 1})")
        print(f"   Active DOFs: {isolated_dof_list}")
        print(f"{'*'*50}")
        
        # 1. Generate the grid safely using DEEPCOPY
        current_grid = copy.deepcopy(ablation_grid_path)
        for idx in range(len(current_grid)):
            # Pass ONLY the single isolated sensor
            current_grid[idx]["dofs"] = isolated_dof_list.copy() 
        
        # 2. Define isolated save locations
        db_isolated, out_isolated, cache_isolated = define_save_locations(
            phase_name,
            dataset=dataset,
            DOFs=isolated_dof_list,
            discretization=discretization
            )
        
        # 3. Execute the standard model pipeline for all 17 architectures
        sweep_results = execute_ablation_pipeline(
            experiment_path=current_grid, 
            database_name=db_isolated, 
            output_dir_name=out_isolated, 
            cache_dir_name=cache_isolated, 
            dataset=dataset,
            n_trials=50,
            epochs=50
        )
        
        # Save results to our master dictionary using the sensor name as the key
        isolated_sweep_master_results[sensor_name] = sweep_results
        
        # 4. Generate the stochastic plots for this specific isolated sensor
        plot_stochastic_summary(
            path_list=current_grid, 
            db_storage=db_isolated, 
            experiment_root_dir=out_isolated, 
            summary_output_dir=os.path.join(out_isolated, "Summary_Plots"),
            file_prefix=f'isolated_sweep_{sensor_name}'
        )
        
        print(f"✅ Finished Isolated Sweep for {sensor_name}.")

except KeyboardInterrupt:
    print(f"Gracefully stopping the Isolated Sweep. Completed {len(isolated_sweep_master_results)} isolated sensors.")
    print("All generated data, databases, and plots up to this point are safely saved!")
    
print("\n🎉 PHASE 5 COMPLETE.")